# Week 4: Direct Corpus Interaction (DCI) for Natural Hazard Literature Search

## Background & Motivation

The original assignment asked for a RAG pipeline (PDF → chunks → embeddings → FAISS → retrieval). This submission replaces that with a **Direct Corpus Interaction (DCI)** approach, adapting the [DCI-Agent-Lite](https://github.com/DCI-Agent/DCI-Agent-Lite) framework for a domain-specific corpus of **Natural Hazard** research papers.

### Why switch from RAG to DCI?

| Aspect | RAG (vector search) | DCI (agent + tools) |
|--------|--------------------|-----------------------|
| Index type | Dense vector index (FAISS) | Pre-extracted sidecar files |
| Retrieval | Approximate nearest-neighbour | Exact text search (`rg`, `jq`) |
| LLM coupling | Reader stage only | Agent drives the whole search loop |
| Transparency | Low (embedding space) | High (all tool calls visible) |
| Embedding drift | Yes — model/domain mismatch | None — no embeddings |
| Infra | Vector DB + embedding server | Local Ollama model only |

For a specialised domain like Natural Hazards (geophysics, seismology, hydrology, remote sensing), generic embedding models often underperform because domain terminology maps poorly into a general-purpose embedding space. DCI sidesteps this by letting the LLM reason directly over structured sidecar files produced by a one-time extraction pipeline.

### Domain selection
Working in the Natural Hazard domain, so the corpus consists of domain-relevant papers extracted by the `PDF_extract` pipeline into `papers_corpus/`.

## Learning Objectives

* Understand why DCI can outperform RAG for specialised scientific corpora.
* Understand the sidecar-file corpus layout produced by `PDF_extract`.
* Configure a local Ollama model as the LLM backend (no cloud API required).
* Run queries via the Streamlit UI and inspect the agent's tool-call traces.

## Architecture Overview

```
PDF papers
    │
    ▼  (PDF_extract pipeline)
papers_corpus/
  {paper_id}/
    manifest.json              ← structured figure/table metadata
    paper_native_images.md     ← full paper text with image references
    tables_structured.md       ← all tables as GFM markdown
    figure_alt_text.md         ← VLM-generated figure descriptions
    │
    ▼
DCI-Agent-Lite (pi harness)
  ├─ tools: read, bash (rg / find / jq)
  ├─ system prompt: pdf_corpus_prompt.txt + image_addendum.txt (written as part of this project)
  └─ provider: Ollama (local LLM)
    │
    ▼
Streamlit UI (app.py)
```

The agent never re-parses PDFs. It uses **ripgrep (`rg`)**, **`find`**, and **`jq`** to search the sidecar files, then synthesises an answer.

## Part 1 — Corpus Layout

In [ ]:
import json
from pathlib import Path
from collections import Counter

CORPUS_DIR = Path("/Users/ming/Desktop/Code/PDF_extract/papers_corpus")

papers = sorted([p.name for p in CORPUS_DIR.iterdir() if p.is_dir()])
print(f"Total papers: {len(papers)}")
for p in papers:
    print(" ", p)

In [ ]:
# Sidecar files for the first paper
first_paper = CORPUS_DIR / papers[0]
print(f"Sidecar files in '{papers[0]}':")
for f in sorted(first_paper.iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:<40} {size_kb:>5} KB")

In [ ]:
# Caption quality breakdown across the whole corpus
quality_counts: Counter = Counter()
for paper_dir in CORPUS_DIR.iterdir():
    mf = paper_dir / "manifest.json"
    if not mf.exists():
        continue
    data = json.loads(mf.read_text(encoding="utf-8"))
    for fig in data.get("figures", []):
        quality_counts[fig.get("caption_quality", "unknown")] += 1

print("Caption quality distribution:")
for quality, count in quality_counts.most_common():
    print(f"  {quality:<12}: {count}")

## Part 2 — System Prompt

Two prompt files were written as part of setting up DCI_search:

- **`prompts/pdf_corpus_prompt.txt`** — documents the corpus layout and `manifest.json` schema for the agent, and sets hard rules (answer the literal question, no hallucinated IDs, cap tool calls at 20) along with recommended tool patterns (`rg`, `jq`, `find`).
- **`prompts/image_addendum.txt`** — appended when "Return figure" is enabled in the UI; adds rules for choosing and returning a `native_img_path` from `manifest.json`.

## Part 3 — Streamlit UI Demo

```bash
cd /Users/ming/Desktop/Code/DCI_search
uv run streamlit run app.py
```

The sidebar lets you pick the Ollama model, toggle figure return, and monitor context-window usage.

---

![DCI Search UI](HW/DCI.png)

The screenshot shows a query *"How many methodology can we use for seismic source location?"* answered by `gemma4:26b` running locally via Ollama. The agent searches the corpus and synthesises an answer citing specific paper IDs (`fuchs-2018`, `li-2024`). With "Return figure" enabled, the agent also resolves a `native_img_path` from the relevant paper's `manifest.json`.

## Source Code

The full DCI_search project is included in `HW/DCI_search/`. Key files:

| File | Description |
|------|-------------|
| `app.py` | Streamlit UI — model selector, figure toggle, chat interface |
| `prompts/pdf_corpus_prompt.txt` | System prompt describing corpus layout and search rules |
| `prompts/image_addendum.txt` | Appended when "Return figure" is enabled |
| `extensions/ollama-provider.ts` | Registers local Ollama models with the pi harness |
| `scripts/regen_ollama_extension.py` | Auto-regenerates the extension file from `ollama list` |
| `scripts/examples/pdf_search_ollama.sh` | CLI entry point |
| `src/dci/benchmark/pi_rpc_runner.py` | Core RPC runner that drives the agent and writes run artifacts |